In [1]:
# ============================================
# OPTIMIZED CNN-TRANSFORMER HYBRID FOR PD DETECTION
# Wavelet + Statistical + Entropy Features ONLY
# Fast & Accurate - Target 80%+ Accuracy
# ============================================

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
import librosa
import pywt
from scipy import stats
from scipy.signal import periodogram
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                            f1_score, roc_auc_score, confusion_matrix,
                            balanced_accuracy_score, cohen_kappa_score,
                            matthews_corrcoef)
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import pickle
import time
import random
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import gc

# Set seeds
def set_all_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_all_seeds(42)

print("="*100)
print("🔬 OPTIMIZED CNN-TRANSFORMER HYBRID")
print("   Wavelet Transform + Statistical Features + Entropy Features")
print("   NO MFCC - Pure Signal Analysis")
print("="*100)

# ============================================
# CONFIGURATION
# ============================================
BASE_PATH = r"E:\miniproject\PD"
OUTPUT_DIR = "pd_transformer_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "graphs"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "models"), exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n📍 Using device: {device}")

class Config:
    TARGET_SR = 16000
    DURATION = 3
    BATCH_SIZE = 64  # Increased for speed
    LEARNING_RATE = 0.001
    EPOCHS = 150  # More epochs for better convergence
    TEST_SIZE = 0.2
    VAL_SIZE = 0.1
    WAVELET_LEVEL = 3
    WAVELET_NAME = 'db4'
    NUM_WORKERS = 0
    
config = Config()

# ============================================
# DATA LOADER
# ============================================
def load_data(base_path):
    """Load all audio files"""
    base_path = Path(base_path)
    file_list = []
    vowels = ['a', 'e', 'i', 'o', 'u', 'apto', 'petaka']
    
    print(f"\n📁 Scanning: {base_path}")
    
    for vowel in vowels:
        vowel_path = base_path / vowel
        if not vowel_path.exists():
            continue
        
        # Healthy files
        healthy_path = vowel_path / "Healthy"
        if healthy_path.exists():
            for f in healthy_path.glob('*.wav'):
                file_list.append({'path': str(f), 'label': 0, 'vowel': vowel})
        
        # PD files
        pd_path = vowel_path / "PD"
        if pd_path.exists():
            for f in pd_path.glob('*.wav'):
                file_list.append({'path': str(f), 'label': 1, 'vowel': vowel})
    
    print(f"\n✅ Total files: {len(file_list)}")
    
    # Show distribution
    for vowel in vowels:
        vowel_files = [f for f in file_list if f['vowel'] == vowel]
        if vowel_files:
            pd_count = sum(1 for f in vowel_files if f['label'] == 1)
            print(f"   {vowel:10s}: {len(vowel_files):3d} files (PD: {pd_count:3d}, Healthy: {len(vowel_files)-pd_count:3d})")
    
    return file_list

# ============================================
# FAST AUDIO LOADING
# ============================================
def load_audio_fast(file_path, target_sr=16000, duration=3):
    """Fast audio loading"""
    try:
        signal, sr = librosa.load(file_path, sr=target_sr, duration=duration, mono=True)
        target_len = target_sr * duration
        if len(signal) < target_len:
            signal = np.pad(signal, (0, target_len - len(signal)))
        elif len(signal) > target_len:
            signal = signal[:target_len]
        
        # Normalize
        max_val = np.max(np.abs(signal))
        if max_val > 0:
            signal = signal / max_val
        
        return signal
    except:
        return None

# ============================================
# PURE FEATURE EXTRACTION (NO MFCC)
# ============================================
def extract_features_pure(file_path):
    """Extract features using ONLY wavelet + statistical + entropy (NO MFCC)"""
    try:
        # Load audio
        signal = load_audio_fast(file_path, config.TARGET_SR, config.DURATION)
        if signal is None:
            return None
        
        features = []
        
        # ========== 1. WAVELET FEATURES ==========
        try:
            coeffs = pywt.wavedec(signal, config.WAVELET_NAME, level=config.WAVELET_LEVEL)
            for i, c in enumerate(coeffs):
                if len(c) > 0:
                    # Energy
                    features.append(np.sum(c**2) / len(c))
                    # Entropy of coefficients
                    hist, _ = np.histogram(c, bins=20)
                    hist = hist[hist > 0]
                    probs = hist / (np.sum(hist) + 1e-10)
                    features.append(-np.sum(probs * np.log2(probs + 1e-10)))
                    # Mean absolute
                    features.append(np.mean(np.abs(c)))
                    # Standard deviation
                    features.append(np.std(c))
                    # Maximum absolute
                    features.append(np.max(np.abs(c)))
        except:
            features.extend([0.0] * (5 * (config.WAVELET_LEVEL + 1)))
        
        # ========== 2. STATISTICAL FEATURES ==========
        # Basic stats
        features.append(np.mean(signal))
        features.append(np.std(signal))
        features.append(np.var(signal))
        features.append(stats.skew(signal))
        features.append(stats.kurtosis(signal))
        
        # Min, Max, Range
        features.append(np.min(signal))
        features.append(np.max(signal))
        features.append(np.ptp(signal))
        
        # Quartiles
        q1, q2, q3 = np.percentile(signal, [25, 50, 75])
        features.append(q1)
        features.append(q2)
        features.append(q3)
        features.append(q3 - q1)  # IQR
        
        # Energy, Power, RMS
        energy = np.sum(signal**2)
        features.append(energy)
        features.append(energy / len(signal))
        features.append(np.sqrt(energy / len(signal)))
        
        # Zero-crossing rate
        zcr = np.sum(np.abs(np.diff(np.sign(signal)))) / (2 * len(signal))
        features.append(zcr)
        
        # ========== 3. ENTROPY FEATURES ==========
        # Spectral Entropy
        try:
            f, Pxx = periodogram(signal, config.TARGET_SR, nfft=512)
            Pxx_norm = Pxx / (np.sum(Pxx) + 1e-10)
            Pxx_norm = Pxx_norm[Pxx_norm > 0]
            features.append(-np.sum(Pxx_norm * np.log2(Pxx_norm + 1e-10)))
        except:
            features.append(0.0)
        
        # Sample Entropy (simplified)
        try:
            m = 2
            r = 0.2 * np.std(signal)
            n = len(signal)
            if n > 100:
                patterns = np.array([signal[i:i+m] for i in range(n - m)])
                distances = np.max(np.abs(patterns[:, None, :] - patterns[None, :, :]), axis=2)
                count = np.sum((distances < r) & (~np.eye(len(patterns), dtype=bool)))
                if count > 0:
                    features.append(-np.log(count / (len(patterns) * (len(patterns)-1))))
                else:
                    features.append(0.0)
            else:
                features.append(0.0)
        except:
            features.append(0.0)
        
        # Approximate Entropy
        try:
            m = 2
            r = 0.2 * np.std(signal)
            n = len(signal)
            if n > 100:
                patterns = np.array([signal[i:i+m] for i in range(n - m + 1)])
                distances = np.max(np.abs(patterns[:, None, :] - patterns[None, :, :]), axis=2)
                C = np.mean(distances <= r, axis=1)
                features.append(np.mean(np.log(C + 1e-10)))
            else:
                features.append(0.0)
        except:
            features.append(0.0)
        
        # Permutation Entropy
        try:
            m = 3
            n = len(signal)
            if n > 100:
                permutations = []
                for i in range(n - m + 1):
                    pattern = signal[i:i+m]
                    perm = tuple(np.argsort(pattern))
                    permutations.append(perm)
                unique, counts = np.unique(permutations, return_counts=True)
                probs = counts / len(permutations)
                features.append(-np.sum(probs * np.log2(probs + 1e-10)))
            else:
                features.append(0.0)
        except:
            features.append(0.0)
        
        return np.array(features, dtype=np.float32)
        
    except Exception as e:
        return None

# ============================================
# OPTIMIZED CNN-TRANSFORMER MODEL
# ============================================
class OptimizedCNNTransformer(nn.Module):
    def __init__(self, input_dim, num_classes=2):
        super().__init__()
        
        # CNN Feature Extractor
        self.cnn = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
        )
        
        # Transformer with fewer parameters
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=128, 
                nhead=4, 
                dim_feedforward=256,
                dropout=0.2,
                activation='gelu',
                batch_first=True
            ),
            num_layers=2
        )
        
        # Classification Head
        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(32, num_classes)
        )
        
    def forward(self, x):
        # CNN extraction
        x = self.cnn(x)
        
        # Add sequence dimension
        x = x.unsqueeze(1)
        
        # Transformer
        x = self.transformer(x)
        
        # Global pooling
        x = x.mean(dim=1)
        
        # Classification
        x = self.classifier(x)
        
        return x

# ============================================
# TRAINING WITH BATCH NORM FIX
# ============================================
def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for data, target in train_loader:
        data, target = data.to(device), target.to(device)
        
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = output.max(1)
        total += target.size(0)
        correct += predicted.eq(target).sum().item()
    
    return running_loss / len(train_loader), 100. * correct / total

def validate(model, val_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_targets = []
    all_probs = []
    
    with torch.no_grad():
        for data, target in val_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = criterion(output, target)
            
            running_loss += loss.item()
            _, predicted = output.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(target.cpu().numpy())
            all_probs.extend(F.softmax(output, dim=1).cpu().numpy())
    
    return running_loss / len(val_loader), 100. * correct / total, all_preds, all_targets, all_probs

def train_model_optimized(X_train, y_train, X_val, y_val, input_dim):
    """Optimized training with better convergence"""
    
    # Handle class imbalance
    class_counts = np.bincount(y_train)
    class_weights = 1.0 / class_counts
    sample_weights = class_weights[y_train]
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights))
    
    train_dataset = TensorDataset(torch.FloatTensor(X_train), torch.LongTensor(y_train))
    val_dataset = TensorDataset(torch.FloatTensor(X_val), torch.LongTensor(y_val))
    
    train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, sampler=sampler)
    val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE)
    
    model = OptimizedCNNTransformer(input_dim).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.LEARNING_RATE, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=15, factor=0.5)
    
    best_val_acc = 0
    best_model_state = None
    patience_counter = 0
    
    print("\n   Training Progress:")
    print("   " + "-" * 60)
    
    for epoch in range(config.EPOCHS):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc, _, _, _ = validate(model, val_loader, criterion, device)
        
        scheduler.step(val_loss)
        
        if (epoch + 1) % 15 == 0:
            print(f"      Epoch {epoch+1:3d}: Train Acc={train_acc:.2f}%, Val Acc={val_acc:.2f}%")
        
        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
        
        # Early stopping
        if patience_counter >= 20:
            print(f"      Early stopping at epoch {epoch+1}")
            break
    
    # Load best model
    if best_model_state:
        model.load_state_dict(best_model_state)
        print(f"\n   ✅ Best validation accuracy: {best_val_acc:.2f}%")
    
    return model, best_val_acc

# ============================================
# VOWEL-WISE EVALUATION
# ============================================
def evaluate_vowel_wise(model, X_test, y_test, vowel_test, scaler, device):
    """Evaluate model for each vowel"""
    vowel_results = {}
    vowel_order = ['a', 'e', 'i', 'o', 'u', 'apto', 'petaka']
    
    print("\n" + "="*100)
    print("📊 VOWEL-WISE CLASSIFICATION RESULTS")
    print("="*100)
    print(f"\n{'Vowel':<12} {'Accuracy':<18} {'Precision':<12} {'Recall':<12} {'F1':<12} {'Samples'}")
    print("-"*85)
    
    for vowel in vowel_order:
        mask = vowel_test == vowel
        if not np.any(mask):
            continue
        
        X_vowel = X_test[mask]
        y_vowel = y_test[mask]
        X_vowel_scaled = scaler.transform(X_vowel)
        
        model.eval()
        with torch.no_grad():
            X_tensor = torch.FloatTensor(X_vowel_scaled).to(device)
            output = model(X_tensor)
            _, predictions = output.max(1)
            y_pred = predictions.cpu().numpy()
        
        # Metrics
        acc = accuracy_score(y_vowel, y_pred)
        precision = precision_score(y_vowel, y_pred, zero_division=0)
        recall = recall_score(y_vowel, y_pred, zero_division=0)
        f1 = f1_score(y_vowel, y_pred, zero_division=0)
        
        # Bootstrap for std
        bootstrap_accs = []
        for _ in range(100):
            idx = np.random.choice(len(y_vowel), len(y_vowel), replace=True)
            boot_acc = accuracy_score(y_vowel[idx], y_pred[idx])
            bootstrap_accs.append(boot_acc)
        std = np.std(bootstrap_accs)
        
        vowel_results[vowel] = {
            'accuracy': acc,
            'std': std,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'n_samples': len(X_vowel),
            'pd_count': np.sum(y_vowel),
            'healthy_count': len(X_vowel) - np.sum(y_vowel)
        }
        
        print(f"{vowel:<12} {acc*100:.2f}% ± {std*100:.2f}%   {precision*100:.2f}%      {recall*100:.2f}%      {f1:.4f}      {len(X_vowel)}")
    
    return vowel_results

# ============================================
# PLOTTING
# ============================================
def plot_confusion_matrix(cm, save_path=None):
    plt.figure(figsize=(8, 6))
    cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
    
    annot = np.empty_like(cm).astype(str)
    for i in range(2):
        for j in range(2):
            annot[i, j] = f'{cm[i, j]}\n({cm_percent[i, j]:.1f}%)'
    
    sns.heatmap(cm, annot=annot, fmt='', cmap='Blues',
                xticklabels=['Healthy', 'PD'],
                yticklabels=['Healthy', 'PD'])
    plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

def plot_vowel_results(vowel_results, save_path=None):
    plt.figure(figsize=(12, 6))
    
    vowels = list(vowel_results.keys())
    accuracies = [vowel_results[v]['accuracy'] * 100 for v in vowels]
    stds = [vowel_results[v]['std'] * 100 for v in vowels]
    
    bars = plt.bar(vowels, accuracies, yerr=stds, capsize=8, 
                   color='#2E86AB', edgecolor='black', linewidth=1.5)
    
    for bar, acc in zip(bars, accuracies):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                f'{acc:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    plt.title('Vowel-wise Classification Accuracy', fontsize=16, fontweight='bold')
    plt.xlabel('Vowel', fontsize=13)
    plt.ylabel('Accuracy (%)', fontsize=13)
    plt.ylim(0, 105)
    plt.grid(axis='y', alpha=0.3)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

def plot_metrics(metrics, save_path=None):
    plt.figure(figsize=(10, 6))
    keys = ['Accuracy', 'Balanced\nAccuracy', 'Precision', 'Recall', 'F1', 'Specificity']
    values = [
        metrics['accuracy'] * 100,
        metrics['balanced_accuracy'] * 100,
        metrics['precision'] * 100,
        metrics['recall'] * 100,
        metrics['f1'] * 100,
        metrics.get('specificity', 0) * 100
    ]
    
    plt.bar(keys, values, color='#2E86AB', edgecolor='black', linewidth=1.5)
    plt.title('Performance Metrics', fontsize=14, fontweight='bold')
    plt.ylabel('Score (%)')
    plt.ylim(0, 105)
    plt.grid(axis='y', alpha=0.3)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

# ============================================
# MAIN PIPELINE
# ============================================
def main():
    print("\n" + "="*100)
    print("🚀 STARTING OPTIMIZED CNN-TRANSFORMER PIPELINE")
    print("="*100)
    
    start_time = time.time()
    
    # Step 1: Load data
    print("\n[1/5] Loading data...")
    file_list = load_data(BASE_PATH)
    
    if len(file_list) == 0:
        print("\n❌ ERROR: No files found!")
        return None
    
    # Step 2: Extract features (NO MFCC)
    print("\n[2/5] Extracting pure features (Wavelet + Statistical + Entropy)...")
    print("   NO MFCC - Using only signal-based features")
    print("   Processing 1700 files...")
    
    features_list = []
    labels_list = []
    vowel_list = []
    
    for file_info in tqdm(file_list, desc="   Extracting", unit="files"):
        features = extract_features_pure(file_info['path'])
        if features is not None:
            features_list.append(features)
            labels_list.append(file_info['label'])
            vowel_list.append(file_info['vowel'])
    
    X = np.array(features_list)
    y = np.array(labels_list)
    vowels = np.array(vowel_list)
    
    print(f"\n   ✅ Features extracted: {len(features_list)} files")
    print(f"   📊 Feature dimension: {X.shape[1]}")
    
    # Step 3: Split data
    print("\n[3/5] Splitting data...")
    
    X_temp, X_test, y_temp, y_test, v_temp, v_test = train_test_split(
        X, y, vowels, test_size=config.TEST_SIZE, random_state=42, stratify=y
    )
    
    X_train, X_val, y_train, y_val, v_train, v_val = train_test_split(
        X_temp, y_temp, v_temp, test_size=config.VAL_SIZE/(1-config.TEST_SIZE), 
        random_state=42, stratify=y_temp
    )
    
    print(f"\n   📊 Final split:")
    print(f"      Train: {len(X_train)} (PD: {np.sum(y_train)}, Healthy: {len(X_train)-np.sum(y_train)})")
    print(f"      Validation: {len(X_val)} (PD: {np.sum(y_val)}, Healthy: {len(X_val)-np.sum(y_val)})")
    print(f"      Test: {len(X_test)} (PD: {np.sum(y_test)}, Healthy: {len(X_test)-np.sum(y_test)})")
    
    # Step 4: Scale features
    print("\n[4/5] Scaling features...")
    scaler = RobustScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)
    
    # Step 5: Train model
    print("\n[5/5] Training Optimized CNN-Transformer Model...")
    model, best_val_acc = train_model_optimized(
        X_train_scaled, y_train, X_val_scaled, y_val, X.shape[1]
    )
    
    # Final evaluation
    print("\n" + "="*100)
    print("📊 FINAL TEST SET RESULTS")
    print("="*100)
    
    test_dataset = TensorDataset(torch.FloatTensor(X_test_scaled), torch.LongTensor(y_test))
    test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE)
    
    _, test_acc, y_pred, y_true, y_proba = validate(model, test_loader, nn.CrossEntropyLoss(), device)
    
    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred)
    balanced_acc = balanced_accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)
    roc_auc = roc_auc_score(y_true, y_proba[:, 1])
    
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    metrics_dict = {
        'accuracy': accuracy,
        'balanced_accuracy': balanced_acc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'specificity': specificity,
        'cohen_kappa': kappa,
        'mcc': mcc,
        'roc_auc': roc_auc,
        'confusion_matrix': cm
    }
    
    print(f"\n🎯 Accuracy:           {accuracy*100:.2f}%")
    print(f"🎯 Balanced Accuracy:  {balanced_acc*100:.2f}%")
    print(f"🎯 Precision:          {precision*100:.2f}%")
    print(f"🎯 Recall:             {recall*100:.2f}%")
    print(f"🎯 F1-Score:           {f1:.4f}")
    print(f"🎯 Specificity:        {specificity*100:.2f}%")
    print(f"🎯 Cohen's Kappa:      {kappa:.4f}")
    print(f"🎯 MCC:                {mcc:.4f}")
    print(f"🎯 ROC-AUC:            {roc_auc:.4f}")
    
    print(f"\n📊 Confusion Matrix:")
    print(f"               Predicted")
    print(f"               Healthy   PD")
    print(f"Actual Healthy   {cm[0,0]:3d}     {cm[0,1]:3d}")
    print(f"       PD        {cm[1,0]:3d}     {cm[1,1]:3d}")
    
    # Vowel-wise evaluation
    vowel_results = evaluate_vowel_wise(model, X_test_scaled, y_test, v_test, scaler, device)
    
    # Generate plots
    print("\n📈 Generating visualizations...")
    plot_confusion_matrix(cm, os.path.join(OUTPUT_DIR, 'graphs', 'confusion_matrix.png'))
    plot_metrics(metrics_dict, os.path.join(OUTPUT_DIR, 'graphs', 'metrics.png'))
    plot_vowel_results(vowel_results, os.path.join(OUTPUT_DIR, 'graphs', 'vowel_results.png'))
    
    # Save results
    vowel_df = pd.DataFrame([
        {
            'Vowel': v,
            'Accuracy': vr['accuracy'],
            'Std': vr['std'],
            'Accuracy_%': f"{vr['accuracy']*100:.2f}",
            'Precision': vr['precision'],
            'Recall': vr['recall'],
            'F1': vr['f1'],
            'Samples': vr['n_samples'],
            'PD_Count': vr['pd_count'],
            'Healthy_Count': vr['healthy_count']
        }
        for v, vr in vowel_results.items()
    ])
    vowel_df.to_csv(os.path.join(OUTPUT_DIR, 'vowel_results.csv'), index=False)
    
    # Save model and scaler
    torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, 'models', 'transformer_model.pth'))
    with open(os.path.join(OUTPUT_DIR, 'scaler.pkl'), 'wb') as f:
        pickle.dump(scaler, f)
    
    # Save results
    results = {
        'metrics': metrics_dict,
        'vowel_results': vowel_results,
        'best_val_acc': best_val_acc,
        'total_time': time.time() - start_time
    }
    
    with open(os.path.join(OUTPUT_DIR, 'results.pkl'), 'wb') as f:
        pickle.dump(results, f)
    
    print("\n" + "="*100)
    print("✅ PIPELINE COMPLETED SUCCESSFULLY!")
    print("="*100)
    print(f"\n📁 Results saved to: {OUTPUT_DIR}/")
    print(f"⏱️  Total time: {(time.time()-start_time)/60:.2f} minutes")
    
    return results, vowel_results

# ============================================
# RUN
# ============================================
if __name__ == "__main__":
    gc.collect()
    results, vowel_results = main()
    
    print(f"\n🎯 Final Test Accuracy: {results['metrics']['accuracy']*100:.2f}%")
    print(f"🎯 Mean Vowel Accuracy: {np.mean([vr['accuracy'] for vr in vowel_results.values()])*100:.2f}%")

🔬 OPTIMIZED CNN-TRANSFORMER HYBRID
   Wavelet Transform + Statistical Features + Entropy Features
   NO MFCC - Pure Signal Analysis

📍 Using device: cpu

🚀 STARTING OPTIMIZED CNN-TRANSFORMER PIPELINE

[1/5] Loading data...

📁 Scanning: E:\miniproject\PD

✅ Total files: 1700
   a         : 300 files (PD: 150, Healthy: 150)
   e         : 300 files (PD: 150, Healthy: 150)
   i         : 300 files (PD: 150, Healthy: 150)
   o         : 300 files (PD: 150, Healthy: 150)
   u         : 300 files (PD: 150, Healthy: 150)
   apto      : 100 files (PD:  50, Healthy:  50)
   petaka    : 100 files (PD:  50, Healthy:  50)

[2/5] Extracting pure features (Wavelet + Statistical + Entropy)...
   NO MFCC - Using only signal-based features
   Processing 1700 files...


   Extracting: 100%|██████████| 1700/1700 [21:42<00:00,  1.31files/s]



   ✅ Features extracted: 1700 files
   📊 Feature dimension: 40

[3/5] Splitting data...

   📊 Final split:
      Train: 1190 (PD: 595, Healthy: 595)
      Validation: 170 (PD: 85, Healthy: 85)
      Test: 340 (PD: 170, Healthy: 170)

[4/5] Scaling features...

[5/5] Training Optimized CNN-Transformer Model...

   Training Progress:
   ------------------------------------------------------------
      Epoch  15: Train Acc=74.54%, Val Acc=74.71%
      Epoch  30: Train Acc=77.31%, Val Acc=71.76%
      Epoch  45: Train Acc=78.57%, Val Acc=75.29%
      Early stopping at epoch 48

   ✅ Best validation accuracy: 78.24%

📊 FINAL TEST SET RESULTS


TypeError: list indices must be integers or slices, not tuple